In [14]:
import pandas as pd
import numpy as np

MISSING_VALUES = ['...', '..', '…']


In [10]:
## Cleaning and unpivoting "Percentage of municipal waste collected which is recycled.xlsx" excel file

df_rec = pd.read_excel("../raw/unsd/Percentage of municipal waste collected which is recycled.xlsx", sheet_name="Data")

# 1. Keeping actual country rows and dropping the blank separator and Footnotes columns
df_rec = df_rec.iloc[:105].loc[:, :2024].copy()

# 2. Converting Country IDs to integer, standardizing country names and source
df_rec['CountryID'] = pd.to_numeric(df_rec['CountryID']).astype('Int64')
df_rec['Country'] = df_rec['Country'].str.strip()
df_rec['Source'] = df_rec['Source'].str.strip()

# Fix missing CountryID for Kosovo* (it has no country code in the excel file)
df_rec.loc[df_rec['Country'] == 'Kosovo*', 'CountryID'] = 383

# 3. Replacing missing values with NaN and converting year columns to numeric
year_cols_rec = [c for c in df_rec.columns if isinstance(c, int)]
df_rec[year_cols_rec] = (df_rec[year_cols_rec].replace(MISSING_VALUES, np.nan).apply(pd.to_numeric))

# 4. Unpivoting the table and making it one row per country and year
df_rec_long = df_rec.melt(
    id_vars=['CountryID', 'Country', 'Source'],
    value_vars=year_cols_rec,
    var_name='Year',
    value_name='Recycled_Pct'
)

# 5. Converting year values to integer
df_rec_long['Year'] = df_rec_long['Year'].astype('Int64')

print("Recycled shape:", df_rec_long.shape)
df_rec_long.head()


Recycled shape: (2520, 5)


,CountryID,Country,Source,Year,Recycled_Pct
0,8,Albania,E,2001,8.352477
1,20,Andorra,U,2001,0.000000
2,660,Anguilla,U,2001,NaN
3,32,Argentina,U,2001,NaN
4,36,Australia,O,2001,NaN


In [9]:
## Cleaing and unpivoting "Municipal waste collected.xlsx" excel file

df_col = pd.read_excel("../raw/unsd/Municipal waste collected.xlsx", sheet_name="Data")

# 1. Keeping actual country rows and dropping the blank separator and Footnotes columns
df_col = df_col.iloc[:143].loc[:, :2023].copy()

# 2. Converting Country IDs to integer, standardizing country names and source
df_col['CountryID'] = pd.to_numeric(df_col['CountryID']).astype('Int64')
df_col['Country'] = df_col['Country'].str.strip()
df_col['Source'] = df_col['Source'].str.strip()

# 3. Replacing missing values with NaN and convert year columns to numeric
year_cols_col = [c for c in df_col.columns if isinstance(c, int)]
df_col[year_cols_col] = (df_col[year_cols_col].replace(MISSING_VALUES, np.nan).apply(pd.to_numeric))

# Fixing broken country code values in the 10 VLOOKUP rows + Russia (no country code) in Municipal waste collected.xlsx
FIXED_COUNTRY_IDS = {
    "Argentina": 32,
    "Australia": 36,
    "Chile": 152,
    "Colombia": 170,
    "Costa Rica": 188,
    "Israel": 376,
    "Japan": 392,
    "Mexico": 484,
    "New Zealand": 554,
    "Saudi Arabia": 682,
    "Russia": 643,
}

# Coerce any '#N/A' or formula errors to NaN, then fill from the lookup dictionary
df_col["CountryID"] = pd.to_numeric(df_col["CountryID"], errors="coerce")
df_col["CountryID"] = (
    df_col["Country"].map(FIXED_COUNTRY_IDS)
    .fillna(df_col["CountryID"])
    .astype("Int64")
)

# Verify zero missing or invalid CountryIDs remain
assert df_col["CountryID"].isna().sum() == 0, "Missing CountryIDs remain!"


# 4. Unpivoting the table and making it one row per country and year (values are in 1000 tonnes)
df_col_long = df_col.melt(
    id_vars=['CountryID', 'Country', 'Source'],
    value_vars=year_cols_col,
    var_name='Year',
    value_name='Waste_Collected_1000t'
)

# 5. Converting year values to integer
df_col_long['Year'] = df_col_long['Year'].astype('Int64')

print("Collected shape:", df_col_long.shape)
df_col_long.head()


Collected shape: (3289, 5)


/var/folders/hv/6m_c5fn10339kvyldbkwm4jr0000gn/T/ipykernel_5049/1784098900.py:36: Pandas4Warning: 'IntegerArray' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  .fillna(df_col["CountryID"])


,CountryID,Country,Source,Year,Waste_Collected_1000t
0,8,Albania,E,2001,NaN
1,20,Andorra,U,2001,37.400000
2,24,Angola,U,2001,NaN
3,660,Anguilla,U,2001,5.634637
4,28,Antigua and Barbuda,U,2001,NaN


In [11]:
## Cleaning and unpivoting "Municipal waste treatment (latest year).xlsx" excel file

df_trt = pd.read_excel("../raw/unsd/Municipal waste treatment (latest year).xlsx", sheet_name="Data")

# 1. Keeping actual country rows and dropping the blank separator and Footnotes rows
#    (skipping the first row because it contains unit labels such as "%" and "1000 tonnes")
df_trt = df_trt.iloc[1:107].copy()

# 2. Dropping the footnotes columns inbetween actual data columns and stripping '\n' newline characters from column names
df_trt = df_trt.loc[:, ~df_trt.columns.str.contains('Footnotes', case=False)]
df_trt.columns = df_trt.columns.str.replace('\n', '', regex=False).str.strip()

# 2. Converting Country IDs and years to integer, standardizing country names and source
df_trt['CountryID'] = pd.to_numeric(df_trt['CountryID']).astype('Int64')
df_trt['Year'] = pd.to_numeric(df_trt['Year']).astype('Int64')
df_trt['Country'] = df_trt['Country'].str.strip()
df_trt['Source'] = df_trt['Source'].str.strip()

# Fixing wrong CountryID (was 826) for Bosnia and Herzegovina in the excel file
df_trt.loc[df_trt['Country'] == 'Bosnia and Herzegovina', 'CountryID'] = 70

# 3. Replacing missing values with NaN and convert year columns to numeric
treatment_cols = [c for c in df_trt.columns if c not in ['CountryID', 'Country', 'Source', 'Year']]
df_trt[treatment_cols] = (df_trt[treatment_cols].replace(MISSING_VALUES, np.nan).apply(pd.to_numeric))

# 5. Unpivoting treatment categories into long format
df_trt_long = df_trt.melt(
    id_vars=['CountryID', 'Country', 'Source', 'Year'],
    value_vars=treatment_cols,
    var_name='Treatment_Type',
    value_name='Value'
)

df_trt_long['Unit'] = np.where(
    df_trt_long['Treatment_Type'] == 'Municipal waste managed',
    '1000 tonnes',
    '%'
)

print("Treatment shape:", df_trt_long.shape)
df_trt_long.head(8)


Treatment shape: (636, 7)


,CountryID,Country,Source,Year,Treatment_Type,Value,Unit
0,8,Albania,E,2023,Municipal waste managed,844.000000,1000 tonnes
1,12,Algeria,U,2021,Municipal waste managed,6762.738800,1000 tonnes
2,20,Andorra,U,2023,Municipal waste managed,36.810000,1000 tonnes
3,660,Anguilla,U,2008,Municipal waste managed,14.654700,1000 tonnes
4,28,Antigua and Barbuda,U,2021,Municipal waste managed,104.530000,1000 tonnes
5,32,Argentina,U,2022,Municipal waste managed,17360.000000,1000 tonnes
6,51,Armenia,U,2023,Municipal waste managed,433.300000,1000 tonnes
7,36,Australia,O,2023,Municipal waste managed,13480.508928,1000 tonnes


In [15]:
## Saving the cleaned tables in CSV files

import os

os.makedirs("../processed", exist_ok=True)

df_col_long.to_csv("../processed/municipal_waste_collected_clean.csv", index=False)
df_rec_long.to_csv("../processed/municipal_waste_recycled_pct_clean.csv", index=False)
df_trt_long.to_csv("../processed/municipal_waste_treatment_latest_clean.csv", index=False)
